In [1]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
mode = 1
"""
mode = 0 ---> non mace
mode = 1 ---> mace
"""
if mode == 0:
    save_win_file_name = 'non_mace'
elif mode == 1:
    save_win_file_name = 'mace'
Zscore = True

In [ ]:
if mode == 0:
    file_dir = r'D:\M143020071\raw_data_result\conversion_data\healthy/'
elif mode == 1:
    file_dir = r'D:\M143020071\raw_data_result\conversion_data\patient/'

hp_id_dir = r'D:\M143020071\raw_data_result/'

if mode == 0:
    if Zscore is True:
        save_dir = r'D:\M143020071\raw_data_result\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\non_mace_zscore/'
    else:
        save_dir = r'D:\M143020071\raw_data_result\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\non_mace/'
elif mode == 1:
    if Zscore is True:
        save_dir = r'D:\M143020071\raw_data_result\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\mace_zscore/'
    else:
        save_dir = r'D:\M143020071\raw_data_result\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\mace/'

In [4]:
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print("Create directory: ", save_dir)
else:
    print("Directory already exists: ", save_dir)

Create directory:  F:\M143020071\raw_data_result\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal_rpeak_5_10min_longer\mace_zscore/


In [5]:
original_sr = 10000
resample_rate = 500
window_size = 10 # unit: second
highcut = 50
lowcut = 0.5

In [6]:
if mode == 0:
    hp_id = pd.read_csv(os.path.join(hp_id_dir, 'MACE_h_id.csv'))
    raw_id = hp_id['research_ID'].values
    covert_id = hp_id['conversion_ID'].values
elif mode == 1:
    hp_id = pd.read_csv(os.path.join(hp_id_dir, 'MACE_p_id.csv'))
    raw_id = hp_id['research_ID'].values
    covert_id = hp_id['conversion_ID'].values

In [7]:
used_5min_ecg_signal = False
win_num_dict = {'research_id':[], 'win_num':[], 'unused_win_num':[]}
for idx, filename in enumerate(tqdm(raw_id, desc="Processing files")):
    # if filename == 594447993595: ## 404479401713 / 594447993595
        # break
    segment_10s_signal_nor_list = []
    segment_10s_signal_list = []
    unused_rpeaks_list = []
    try:
        ecg_signal = pd.read_csv(os.path.join(file_dir, f'{filename}.csv'))
        ecg_signal = ecg_signal['Ch1'].values.astype(np.float32)
        ecg_signal_detrended = nk.signal_detrend(ecg_signal, order=0, method='polynomial', sampling_rate=original_sr)
        ecg_signal_filter_notch = nk.signal_filter(ecg_signal_detrended, sampling_rate=original_sr, method="powerline", powerline=60)
        ecg_signal_filter = nk.signal_filter(ecg_signal_filter_notch, sampling_rate=original_sr, lowcut=lowcut, highcut=highcut, method='butterworth', order=2)
        ecg_signal_resampled = nk.signal_resample(ecg_signal_filter, sampling_rate=original_sr, desired_sampling_rate=resample_rate)
        # print(ecg_signal_resampled.shape)
        if ecg_signal_resampled.shape[0] >= 5 * resample_rate * 60:
            if ecg_signal_resampled.shape[0] < 10 * resample_rate * 60:
                original_offset = ecg_signal_resampled.shape[0] - 5 * resample_rate * 60
                ecg_signal_segment_5min = ecg_signal_resampled[original_offset : ]
            else:
                original_offset = 5 * resample_rate * 60
                end_offset = 10 * resample_rate * 60
                ecg_signal_segment_5min = ecg_signal_resampled[original_offset : end_offset]
        else:
            print(f"File {filename}.csv is too short Skipping.")
            continue
        rpeaks, info = nk.ecg_peaks(ecg_signal_segment_5min, sampling_rate=resample_rate, correct_artifacts=True)
        if used_5min_ecg_signal is True:
            used_ecg_signal = ecg_signal_segment_5min
            signal_offset = 0
        else:
            used_ecg_signal = ecg_signal_resampled
            signal_offset = original_offset
        for i in range(len(info['ECG_R_Peaks'])):
            start_rpeak_index = info['ECG_R_Peaks'][i] + signal_offset
            end_index = start_rpeak_index + resample_rate * window_size
            segment_10s_signal = used_ecg_signal[start_rpeak_index : end_index]
            # ecg_signal_segment_5min : rpeak 起點往後取10秒，如果訊號長度10秒超過10分鐘，則捨棄
            # ecg_signal_resampled : rpeak 起點往後取10秒，即使訊號長度10秒超過10分鐘，仍會使用，但會超出10分鐘範圍
            if segment_10s_signal.shape[0] == resample_rate * window_size:
                if Zscore is True:
                    mean_segment = np.mean(segment_10s_signal)
                    std_segment = np.std(segment_10s_signal)
                    segment_10s_signal_nor = (segment_10s_signal - mean_segment) / std_segment
                    segment_10s_signal_nor_list.append(segment_10s_signal_nor)
                else:
                    segment_10s_signal_list.append(segment_10s_signal)
            else:
                unused_rpeaks_list.append((start_rpeak_index - signal_offset)/resample_rate)

        if Zscore is True:
            segment_10s_signal_nor_arr = np.vstack(segment_10s_signal_nor_list).astype(np.float32) ###
            if mode == 0:
                label_arr = np.zeros((segment_10s_signal_nor_arr.shape[0],1), dtype=np.float32)
            elif mode == 1:
                label_arr = np.ones((segment_10s_signal_nor_arr.shape[0],1), dtype=np.float32)
            segment_10s_signal_nor_arr_with_label = np.hstack((label_arr, segment_10s_signal_nor_arr))
            np.save(os.path.join(save_dir, f'{covert_id[idx]}.npy'), segment_10s_signal_nor_arr_with_label)

            win_num_dict['research_id'].append(f'{covert_id[idx]}')
            win_num_dict['win_num'].append(int(segment_10s_signal_nor_arr.shape[0]))
            win_num_dict['unused_win_num'].append(int(len(unused_rpeaks_list)))

        else:
            segment_10s_signal_arr = np.vstack(segment_10s_signal_list).astype(np.float32)
            if mode == 0:
                label_arr = np.zeros((segment_10s_signal_arr.shape[0],1), dtype=np.float32)
            elif mode == 1:
                label_arr = np.ones((segment_10s_signal_arr.shape[0],1), dtype=np.float32)

            segment_10s_signal_arr_with_label = np.hstack((label_arr, segment_10s_signal_arr))
            np.save(os.path.join(save_dir, f'{covert_id[idx]}.npy'), segment_10s_signal_arr_with_label)

            win_num_dict['research_id'].append(str(f'{covert_id[idx]}'))
            win_num_dict['win_num'].append(int(segment_10s_signal_arr.shape[0]))
            win_num_dict['unused_win_num'].append(int(len(unused_rpeaks_list)))


    except Exception as e:
        print(f"Error reading {filename}.csv: {e}")
        continue

result_df = pd.DataFrame(win_num_dict)
display(result_df)
result_df.to_csv(os.path.join(save_dir, f'win_num_statistic.csv'), index=False)

Processing files: 100%|██████████| 90/90 [05:38<00:00,  3.76s/it]


,research_id,win_num,unused_win_num
0,MACE021,595,0
1,MACE028,344,0
2,MACE033,339,0
3,MACE044,375,0
4,MACE046,352,0
...,...,...,...
85,MACE503,459,0
86,MACE505,410,0
87,MACE507,419,0
88,MACE508,267,0
